## Domo AI Pro Operations & Processing
Domo AI Pro operations and processing will consume credits as described on [Domo's online consumption terms](https://www.domo.com/consumption-terms). Please see the credit rate card in your Domo instance for more information. (Admin > Company Settings > Credit Utilization > Rate Card)

# Tool Calling

`ai.tool_calling()` gives the model a set of tools it can choose to invoke. You define each tool's name, description, and parameter schema. The model decides which tool to call and with what arguments based on the conversation.

## Single tool

Define one tool and ask a question that requires it. The response contains the tool call or a direct answer.

In [ ]:
import domojupyter.ai as ai

lookup_revenue = ai.Tool(
    name="get_revenue",
    description="Look up total revenue for a given region and time period",
    parameters={
        "type": "object",
        "properties": {
            "region": {"type": "string", "description": "The sales region (e.g. North America, EMEA)"},
            "period": {"type": "string", "description": "Time period as a quarter, e.g. Q3 2024"}
        },
        "required": ["region", "period"],
        "additionalProperties": False
    }
)

messages = [{"role": "user", "content": "What was total revenue for EMEA in Q3 2024?"}]

response = ai.tool_calling(messages=messages, tools=[lookup_revenue])
print(response.text)

## Multiple tools

Give the model a toolbox and let it pick the right one. The model may also chain tool calls if needed.

In [ ]:
import domojupyter.ai as ai

get_customer_info = ai.Tool(
    name="get_customer_info",
    description="Retrieve account details for a customer by their ID",
    parameters={
        "type": "object",
        "properties": {
            "customer_id": {"type": "string", "description": "The customer account ID"}
        },
        "required": ["customer_id"],
        "additionalProperties": False
    }
)

get_open_tickets = ai.Tool(
    name="get_open_tickets",
    description="Get the list of open support tickets for a customer",
    parameters={
        "type": "object",
        "properties": {
            "customer_id": {"type": "string", "description": "The customer account ID"},
            "status": {"type": "string", "description": "Filter by ticket status: open, pending, or all"}
        },
        "required": ["customer_id"],
        "additionalProperties": False
    }
)

messages = [{"role": "user", "content": "Check if customer C-10482 has any open support issues"}]

response = ai.tool_calling(
    messages=messages,
    tools=[get_customer_info, get_open_tickets]
)
print(response.text)

## With a system message

Use `system` to set context about what the tools represent or how the model should use them.

In [ ]:
import domojupyter.ai as ai

get_metric = ai.Tool(
    name="get_metric",
    description="Retrieve a business metric value for a given time period",
    parameters={
        "type": "object",
        "properties": {
            "metric_name": {"type": "string", "description": "Name of the metric (e.g. MRR, churn_rate, NPS)"},
            "period": {"type": "string", "description": "Time period, e.g. 2024-Q3 or 2024-10"}
        },
        "required": ["metric_name", "period"],
        "additionalProperties": False
    }
)

messages = [{"role": "user", "content": "What was our MRR and churn rate last quarter?"}]

response = ai.tool_calling(
    messages=messages,
    tools=[get_metric],
    system="You are a business intelligence assistant. Use tools to look up metrics before answering. "
           "Never guess metric values."
)
print(response.text)